Here we will try to make use of the extracted type 2,4,5 data from the rsp_events file.  

In [8]:
import json
import numpy as np

# Load the JSON file
with open("rsp_events.json", "r") as f:
    data = json.load(f)

# Lists to hold each burst type
type_II = []
type_IV = []
type_V = []

# Loop through each event and classify by burst type in "part"
for event in data:
    part = event.get("part", "")
    if part.startswith("II/"):
        type_II.append(event)
    elif part.startswith("IV/"):
        type_IV.append(event)
    elif part.startswith("V/"):
        type_V.append(event)

# Convert to numpy arrays for shape
type_II = np.array(type_II)
type_IV = np.array(type_IV)
type_V = np.array(type_V)

print("Type II shape:", type_II.shape)
print("Type IV shape:", type_IV.shape)
print("Type V shape:", type_V.shape)

Type II shape: (179,)
Type IV shape: (250,)
Type V shape: (285,)


Since most of the data in the offline dataset is not labelled. We will now try to find the find and label the data indexes which are in the offline data from the help of the rsp file. We will save the indexes of the offline data file which does not have corresponding file in the rsp. We will append the h5 file in labels keys with the type from rsp. To append the labels key the condition that the data should not have corresponding label in the h5 file should be met. If the h5 file has label for an image and also for the rsp file, the label would not be replaced. The file will checked via date and time from rsp and timestamp key from the h5 file. The indexes of timestamps for which the labels were appended will also be saved in another variable. But the not found timestamps indexes will be saved in a seperate json file. 

In [15]:
import json

# Load the JSON file
with open("rsp_events.json", "r") as f:
    data = json.load(f)

date_begin_to_part = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        # Format begin as HH:MM:00.000000
        hour = begin[:2]
        minute = begin[2:]
        formatted = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        date_begin_to_part[formatted] = part_clean

# Example: print first 10 entries
for k, v in list(date_begin_to_part.items())[:10]:
    print(f"{k}: {v}")

print("Length of dictionary:", len(date_begin_to_part))

2013-03-19_00:39:00.000000: IV
2013-04-06_05:58:00.000000: V
2013-04-11_09:39:00.000000: V
2013-04-11_10:20:00.000000: IV
2013-04-18_07:59:00.000000: V
2013-04-22_10:26:00.000000: V
2013-04-22_20:44:00.000000: V
2013-04-22_21:16:00.000000: V
2013-04-22_22:20:00.000000: V
2013-04-22_22:40:00.000000: V
Length of dictionary: 697


In [16]:
import json

with open("rsp_events.json", "r") as f:
    data = json.load(f)

key_counts = {}
for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        key_counts[key] = key_counts.get(key, 0) + 1

duplicates = {k: v for k, v in key_counts.items() if v > 1}
print(f"Number of duplicate keys: {len(duplicates)}")
for k, v in duplicates.items():
    print(f"{k}: {v} entries")

Number of duplicate keys: 17
2013-04-22_22:20:00.000000: 2 entries
2013-05-31_19:57:00.000000: 2 entries
2013-06-03_00:00:00.000000: 2 entries
2013-10-28_04:37:00.000000: 2 entries
2014-02-17_03:00:00.000000: 2 entries
2014-07-04_04:38:00.000000: 2 entries
2014-08-01_18:18:00.000000: 2 entries
2014-11-03_03:48:00.000000: 2 entries
2015-06-24_00:00:00.000000: 2 entries
2016-01-01_23:24:00.000000: 2 entries
2016-02-03_23:25:00.000000: 2 entries
2022-04-30_09:57:00.000000: 2 entries
2022-08-27_02:12:00.000000: 2 entries
2022-09-23_13:50:00.000000: 2 entries
2024-01-28_02:28:00.000000: 2 entries
2024-02-06_03:10:00.000000: 2 entries
2024-07-21_01:53:00.000000: 2 entries


The reason you have 714 entries in your JSON file but only 697 unique keys in your dictionary is because some entries have the same combination of date and begin (i.e., the same timestamp key). When you use a dictionary, if two or more events have the same date and begin, the last one will overwrite the previous ones.

Some of the duplicates have same same types but some have different types of burst at the same time instant. Eg, 2013-05-31_19:57:00.000000: 2 entries

Merging part (bursts) of duplicate keys with a "/". These images have 2 bursts for 1.

In [18]:
import json

# Load the JSON file
with open("rsp_events.json", "r") as f:
    data = json.load(f)

roman_to_int = {"II": "2", "IV": "4", "V": "5"}

date_begin_to_parts = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        # Replace Roman numeral with integer
        for roman, integer in roman_to_int.items():
            if part_clean == roman:
                part_clean = integer
        if key in date_begin_to_parts:
            existing = date_begin_to_parts[key].split('/')
            if part_clean not in existing:
                date_begin_to_parts[key] += f"/{part_clean}"
        else:
            date_begin_to_parts[key] = part_clean

# Example: print first 10 entries
for k, v in list(date_begin_to_parts.items())[:10]:
    print(f"{k}: {v}")

print("Length of dictionary:", len(date_begin_to_parts))

2013-03-19_00:39:00.000000: 4
2013-04-06_05:58:00.000000: 5
2013-04-11_09:39:00.000000: 5
2013-04-11_10:20:00.000000: 4
2013-04-18_07:59:00.000000: 5
2013-04-22_10:26:00.000000: 5
2013-04-22_20:44:00.000000: 5
2013-04-22_21:16:00.000000: 5
2013-04-22_22:20:00.000000: 5
2013-04-22_22:40:00.000000: 5
Length of dictionary: 697


In [27]:
import h5py
import re

with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]
    dt_pattern = re.compile(r'\d{4}-\d{2}-\d{2}_\d{2}:\d{2}:\d{2}\.\d+')
    dt_list = []
    for ts in timestamps:
        match = dt_pattern.search(ts.decode())
        if match:
            dt_list.append(match.group(0))
        else:
            dt_list.append(None)

    # Build a mapping from date-time string to index
    dt_map = {dt: i for i, dt in enumerate(dt_list) if dt is not None}

    print("Date-time mapping:", (dt_map))

Date-time mapping: {'2022-05-12_13:00:00.000000': 571, '2022-05-12_13:30:00.000000': 597, '2022-05-12_12:00:00.000000': 581, '2022-05-12_07:00:00.000000': 591, '2022-05-12_07:30:00.000000': 584, '2022-05-12_14:00:00.000000': 578, '2022-05-12_12:30:00.000000': 587, '2022-05-12_11:00:00.000000': 588, '2022-05-12_09:30:00.000000': 602, '2022-05-12_09:00:00.000000': 573, '2022-05-12_11:30:00.000000': 577, '2022-05-12_08:30:00.000000': 592, '2022-05-12_10:00:00.000000': 599, '2022-05-12_10:30:00.000000': 567, '2022-05-12_08:00:00.000000': 583, '2022-06-02_07:00:00.000000': 28, '2022-06-02_12:30:00.000000': 30, '2022-06-02_13:00:00.000000': 32, '2022-06-02_09:00:00.000000': 39, '2022-06-02_10:30:00.000000': 35, '2022-06-02_11:00:00.000000': 37, '2022-06-02_08:30:00.000000': 33, '2022-06-02_10:00:00.000000': 34, '2022-06-02_09:30:00.000000': 38, '2022-06-02_08:00:00.000000': 36, '2022-06-02_11:30:00.000000': 40, '2022-06-02_12:00:00.000000': 29, '2022-06-02_07:30:00.000000': 31, '2022-05-02_1

In [29]:
import h5py
import json
import numpy as np

# Open HDF5 file
# h5_path = '/Volumes/External SSD 512gb/dset_with_labels.h5'
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]  # (8147,)
    labels = file['labels'][:]          # (2854,)
    # Convert labels to a writable array if needed
    labels = np.array(labels)
    # Prepare a mapping from timestamp string (without id) to index
    ts_map = {}
    for i, ts in enumerate(timestamps):
        ts_str = ts.decode().split('_', 1)[1]  # skip id, keep date+time
        ts_map[ts_str] = i

    appended_indices = []
    appended_keys = []
    not_found_keys = []

    label_ptr = 0  # pointer for writing to labels array

    for k, v in date_begin_to_parts.items():
        idx = dt_map.get(k)
        if idx is not None:
            if label_ptr < len(labels):
                # Only assign if label is not already set (0, -1, or nan)
                if labels[label_ptr] == 0 or labels[label_ptr] == -1 or (isinstance(labels[label_ptr], float) and np.isnan(labels[label_ptr])):
                    # If v is a string like "2/4", you may want to handle this as needed
                    try:
                        labels[label_ptr] = int(v.split('/')[0])  # assign first burst type as int
                    except Exception:
                        labels[label_ptr] = -1  # fallback if conversion fails
                    appended_indices.append(label_ptr)
                    appended_keys.append(k)
                # else: label already exists, skip
                label_ptr += 1
            else:
                # No more label slots left
                break
        else:
            not_found_keys.append(k)

    # Save the updated labels back to the file if needed
    # file['labels'][:] = labels  # Uncomment if you want to write back

# Save results
with open("rsp_not_found_keys.json", "w") as f:
    json.dump(not_found_keys, f, indent=2)
with open("rsp_appended_keys.json", "w") as f:
    json.dump(appended_keys, f, indent=2)
with open("rsp_appended_indices.json", "w") as f:
    json.dump(appended_indices, f, indent=2)

print(f"Appended {len(appended_indices)} new labels.")
print(f"Could not find {len(not_found_keys)} keys in timestamps.")

Appended 0 new labels.
Could not find 697 keys in timestamps.


Import csv file for type 2 for labelling